# Lab 05 — Image Segmentation (Python · OpenCV · NumPy)
**How to use**
1. Run the **Setup** cell once.
2. Every task has an `*_URL` variable at the top of its first code cell. Paste the instructor's link there (direct image link, GitHub `blob` link, or Google Drive share link all work). Leave it `""` and Colab will pop up an **upload** button instead. A local path also works.
3. Run the task cells top to bottom. Tune the parameters in the cells marked `# <-- TUNE`.
4. Every figure and final mask is saved to `/content/outputs`. The last cell zips everything.

**Image needed per task**
| Task | Image |
|---|---|
| 1, 2 | Unevenly lit document photo (Task 2 reuses Task 1's) |
| 3 | Grayscale objects on plain background (coins) |
| 4 | Colour image with one clearly coloured object (yellow car) |
| 5 | Object/part with clear + faint edges |
| 6 | Brain MRI |
| 7, 8 | Touching coins (Task 8 reuses Task 7's) |
| 9 | Complex colour photo (wildlife) |
| 10 | Any challenging image (or reuse one above) |

> Only OpenCV + NumPy do the segmentation (no segmentation libraries / deep models). Matplotlib is used for display only.

## Setup

In [ ]:
import cv2, os, re, zipfile
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import requests
from collections import deque

OUT_DIR = "/content/outputs"
os.makedirs(OUT_DIR, exist_ok=True)
plt.rcParams["figure.dpi"] = 110
print("OpenCV", cv2.__version__)

def _fix_url(u):
    u = u.strip()
    m = re.search(r"drive\.google\.com/file/d/([^/]+)", u)
    if m: return f"https://drive.google.com/uc?export=download&id={m.group(1)}"
    m = re.search(r"drive\.google\.com/open\?id=([^&]+)", u)
    if m: return f"https://drive.google.com/uc?export=download&id={m.group(1)}"
    if "github.com" in u and "/blob/" in u:
        return u.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")
    return u

def load_image(src, mode="color"):
    """src = URL / local path / "" (-> upload dialog). mode = 'color' or 'gray'."""
    flag = cv2.IMREAD_COLOR if mode == "color" else cv2.IMREAD_GRAYSCALE
    if src and src.strip():
        if src.strip().startswith("http"):
            r = requests.get(_fix_url(src), headers={"User-Agent": "Mozilla/5.0"}, timeout=40)
            r.raise_for_status()
            data = np.frombuffer(r.content, np.uint8)
        else:
            data = np.fromfile(src.strip(), np.uint8)
    else:
        from google.colab import files
        print("Upload the image for this task:")
        up = files.upload()
        data = np.frombuffer(next(iter(up.values())), np.uint8)
    img = cv2.imdecode(data, flag)
    assert img is not None, "Could not decode image - check the link/file."
    print("Loaded:", img.shape, img.dtype)
    return img

def resize_max(img, m=900):
    h, w = img.shape[:2]
    s = m / max(h, w)
    return img if s >= 1 else cv2.resize(img, (int(w*s), int(h*s)), interpolation=cv2.INTER_AREA)

def show(imgs, titles, cols=None, size=4, suptitle=None, note=None, save=None):
    n = len(imgs); cols = cols or n; rows = int(np.ceil(n/cols))
    fig, axs = plt.subplots(rows, cols, figsize=(size*cols, size*rows*0.95 + (0.9 if note else 0.2)))
    axs = np.atleast_1d(axs).ravel()
    for ax in axs: ax.axis("off")
    for ax, im, t in zip(axs, imgs, titles):
        if im.ndim == 2: ax.imshow(im, cmap="gray", vmin=0, vmax=255)
        else: ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(t, fontsize=9)
    if suptitle: fig.suptitle(suptitle, fontsize=12, fontweight="bold")
    plt.tight_layout()
    if note: fig.text(0.5, 0.0, note, ha="center", va="top", fontsize=9, wrap=True)
    if save: fig.savefig(os.path.join(OUT_DIR, save), bbox_inches="tight", dpi=150)
    plt.show()

def save_img(name, img):
    cv2.imwrite(os.path.join(OUT_DIR, name), img)

def fg_percent(mask): return round(100*float((mask > 0).mean()), 2)

def fix_polarity(mask):
    """Make objects white / background black, using the image border as background."""
    border = np.r_[mask[0], mask[-1], mask[:, 0], mask[:, -1]]
    return 255 - mask if border.mean() > 127 else mask

def noise_specks(mask, max_area=8):
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    return int(np.sum(stats[1:, cv2.CC_STAT_AREA] <= max_area))

---
# Task 1 — Document under uneven lighting
**Image:** unevenly lit document photo. Text is made **white** in the masks (`THRESH_BINARY_INV`) so foreground % is meaningful.

In [ ]:
DOC_URL = ""   # <-- paste the document image link here ("" = upload)

doc = resize_max(load_image(DOC_URL, "gray"))          # 1. grayscale load
H, W = doc.shape

# 2. three global thresholds
T1, T2, T3 = 90, 130, 170                               # <-- TUNE
glob = [cv2.threshold(doc, t, 255, cv2.THRESH_BINARY_INV)[1] for t in (T1, T2, T3)]

# 4. adaptive threshold
ADAPT_BLOCK, ADAPT_C = 31, 10                           # <-- TUNE (block must be odd)
adaptive = cv2.adaptiveThreshold(doc, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv2.THRESH_BINARY_INV, ADAPT_BLOCK, ADAPT_C)

BEST_GLOBAL = 1   # <-- index 0/1/2 of the global result you judge best (after looking at the figure)

In [ ]:
# 3 + 5 + 6: compare masks quantitatively, left vs right side of page
third = W // 3
rows = [("Original (mean brightness)", doc[:, :third].mean(), doc[:, -third:].mean())]
for name, m in zip([f"Global T={T1}", f"Global T={T2}", f"Global T={T3}", "Adaptive"], glob + [adaptive]):
    rows.append((f"{name}  (text % of area)", 100*(m[:, :third] > 0).mean(), 100*(m[:, -third:] > 0).mean()))
df = pd.DataFrame(rows, columns=["Measure", "Left third", "Right third"]).round(2)
display(df)

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot((glob[BEST_GLOBAL] > 0).mean(axis=0), label=f"Best global (T={[T1,T2,T3][BEST_GLOBAL]})")
ax.plot((adaptive > 0).mean(axis=0), label="Adaptive")
ax.set_xlabel("image column (left -> right)"); ax.set_ylabel("fraction of text pixels")
ax.set_title("Where does global thresholding fail? (look for the side where it collapses or floods)")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Required output: Original -> Global T1 -> T2 -> T3 -> Adaptive + the question underneath
q1 = ("Why does a single threshold struggle when illumination changes across the image?\n"
      "Observed intensity = reflectance x illumination. In a dim region even white paper can be darker than ink "
      "in a bright region, so no single cutoff separates text from paper everywhere: a low T loses characters "
      "in the dark side, a high T floods the bright side / paper with foreground. Adaptive thresholding "
      "compares each pixel with its own local neighbourhood, so the local illumination cancels out.")
show([doc] + glob + [adaptive],
     ["Original", f"Global T1={T1}", f"Global T2={T2}", f"Global T3={T3}", f"Adaptive (Gaussian, {ADAPT_BLOCK}, C={ADAPT_C})"],
     cols=5, size=3.6, suptitle="Task 1 - Global vs Adaptive", note=q1, save="task1_comparison.png")
save_img("task1_final_adaptive_mask.png", adaptive)

**Parameters & justification (edit to match your final values)**
- **Global T1/T2/T3 = 90 / 130 / 170** — low, mid, high cutoffs to show both failure modes (missing characters vs flooded paper).
- **Adaptive:** Gaussian-weighted, `blockSize=31`, `C=10` — 31 px neighbourhood is larger than a stroke width (so strokes are not "absorbed" into their own local mean) but small enough to follow the lighting gradient; `C=10` suppresses paper-texture noise.
- **Failure regions:** note which side (bright/dark) your best global mask breaks on, using the column plot above.
- **Most usable mask:** *adaptive* (state your final parameter combination).

---
# Task 2 — Choosing the adaptive strategy (blockSize × C × method)
Reuses the document from Task 1 (set `DOC2_URL` only if the instructor gave a different one).

In [ ]:
DOC2_URL = ""            # <-- leave "" to reuse Task 1's document
doc2 = doc if DOC2_URL == "" and "doc" in globals() else resize_max(load_image(DOC2_URL, "gray"))

BLOCKS = (11, 31, 71)    # <-- TUNE (odd)
CS     = (2, 10, 25)     # <-- TUNE
METHODS = {"Mean": cv2.ADAPTIVE_THRESH_MEAN_C, "Gaussian": cv2.ADAPTIVE_THRESH_GAUSSIAN_C}

res = {}
for mname, mflag in METHODS.items():
    for b in BLOCKS:
        for c in CS:
            res[(mname, b, c)] = cv2.adaptiveThreshold(doc2, 255, mflag, cv2.THRESH_BINARY_INV, b, c)

# parameter comparison table
tab = pd.DataFrame([{"method": k[0], "blockSize": k[1], "C": k[2],
                     "text %": fg_percent(m), "noise specks (<=8px)": noise_specks(m)}
                    for k, m in res.items()])
display(tab)

# full grids (3x3 per method)
for mname in METHODS:
    imgs = [res[(mname, b, c)] for b in BLOCKS for c in CS]
    ttl = [f"{mname} b={b} C={c}" for b in BLOCKS for c in CS]
    show(imgs, ttl, cols=len(CS), size=3.4, suptitle=f"Task 2 - {mname} grid (rows: blockSize, cols: C)",
         save=f"task2_grid_{mname.lower()}.png")

In [ ]:
# Required final output: original + >= 6 adaptive results
FINAL_SIX = [("Gaussian", 31, 10), ("Gaussian", 11, 10), ("Gaussian", 71, 10),
             ("Mean", 31, 10), ("Gaussian", 31, 2), ("Gaussian", 31, 25)]   # <-- TUNE
show([doc2] + [res[k] for k in FINAL_SIX],
     ["Original"] + [f"{m} b={b} C={c}" for m, b, c in FINAL_SIX],
     cols=4, size=3.6, suptitle="Task 2 - Original + six adaptive results", save="task2_final_six.png")

BEST = ("Gaussian", 31, 10)   # <-- your pick after inspecting
save_img("task2_best_mask.png", res[BEST])
print("Best combination:", BEST)

**Analysis (verify each claim against your own grid/table, then keep or edit)**
1. **Neighbourhood too small:** the window sits entirely *inside* a thick stroke, so local mean ≈ the pixel itself → stroke interiors drop out (hollow / broken characters) and paper texture becomes speckle noise.
2. **Neighbourhood too large:** the local mean approaches the global mean → uneven-illumination problems return (shadow regions turn black/white), and nearby dark strokes pull the threshold around.
3. **C increased:** threshold = local mean − C, so fewer pixels qualify → cleaner background, but thin strokes erode and characters break. Small C → more noise around letters.
4. **Cleanest combination:** the mid `blockSize` with moderate `C` (e.g. Gaussian, 31, 10) — confirm with the *noise specks* column and by eye.
5. **Better method:** Gaussian is usually smoother (nearby pixels weigh more, so the threshold surface follows lighting without box artefacts); Mean is equally weighted and tends to look blockier. Decide using your table.

---
# Task 3 — Otsu's thresholding (quality control)
**Image:** grayscale objects on plain background (OpenCV watershed coins image).

In [ ]:
COINS3_URL = ""   # <-- coins image link

coins3 = resize_max(load_image(COINS3_URL, "color"))
gray3 = cv2.cvtColor(coins3, cv2.COLOR_BGR2GRAY)          # 1. grayscale

clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
variants = {
    "Original":             gray3,
    "Gaussian blur 5x5":    cv2.GaussianBlur(gray3, (5, 5), 0),
    "Contrast x1.5 (-60)":  cv2.convertScaleAbs(gray3, alpha=1.5, beta=-60),
    "Brightness +40":       cv2.convertScaleAbs(gray3, alpha=1.0, beta=40),
    "CLAHE":                clahe.apply(gray3),
}                                                           # <-- add/remove variants freely

def otsu(gray):
    t, m = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)   # Otsu picks t automatically
    return t, fix_polarity(m)

records = []
for name, g in variants.items():
    t, m = otsu(g)
    records.append({"preprocessing": name, "Otsu threshold": round(t, 1), "object %": fg_percent(m)})
    # Required output: Original -> Histogram -> Otsu mask
    fig, axs = plt.subplots(1, 3, figsize=(13, 3.6))
    axs[0].imshow(g, cmap="gray", vmin=0, vmax=255); axs[0].set_title(f"Original ({name})"); axs[0].axis("off")
    axs[1].hist(g.ravel(), bins=256, range=(0, 255), color="gray")
    axs[1].axvline(t, color="r", ls="--", label=f"Otsu T = {t:.0f}"); axs[1].legend()
    axs[1].set_title("Histogram"); axs[1].set_xlabel("intensity")
    axs[2].imshow(m, cmap="gray", vmin=0, vmax=255); axs[2].set_title("Otsu binary mask"); axs[2].axis("off")
    plt.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, f"task3_{name.split()[0].lower()}.png"), bbox_inches="tight", dpi=150)
    plt.show()

display(pd.DataFrame(records))
save_img("task3_otsu_mask_original.png", otsu(gray3)[1])

**Why is Otsu useful when the correct threshold isn't known beforehand?**
Otsu scans every possible threshold and keeps the one that **maximises between-class variance** (equivalently minimises within-class variance) of the two pixel groups, so a bimodal histogram (object + background) is split at the valley automatically. The engineer does not hand-tune a value per image, and the threshold adapts when lighting/contrast shifts (compare the recorded thresholds in the table: contrast/brightness changes move T, and the mask stays similar). It works best when the histogram really is two-class; it degrades with uneven lighting or heavy noise (blur first).

---
# Task 4 — Colour-based segmentation in HSV
**Image:** colour image with a distinct coloured object (yellow car). OpenCV HSV ranges: **H 0–179, S 0–255, V 0–255**. Yellow ≈ H 20–35.

In [ ]:
COLOR_URL = ""   # <-- yellow-car (or similar) image link

bgr4 = resize_max(load_image(COLOR_URL, "color"))
hsv4 = cv2.cvtColor(bgr4, cv2.COLOR_BGR2HSV)               # 2. HSV

# Helper: probe pixels on the object to choose your range
PROBES = [(int(bgr4.shape[1]*0.5), int(bgr4.shape[0]*0.5))]   # <-- (x, y) points ON the object
for (x, y) in PROBES:
    print(f"pixel (x={x}, y={y}) HSV =", hsv4[y, x])

fig, axs = plt.subplots(1, 2, figsize=(12, 3.6))
axs[0].imshow(cv2.cvtColor(bgr4, cv2.COLOR_BGR2RGB)); axs[0].set_title("Original (axes = pixel coords for probing)")
for (x, y) in PROBES: axs[0].plot(x, y, "r+", ms=14)
sat = hsv4[..., 1] > 60
axs[1].hist(hsv4[..., 0][sat].ravel(), bins=180, range=(0, 180), color="orange")
axs[1].set_title("Hue histogram (saturated pixels) - find the object's hue peak"); axs[1].set_xlabel("H")
plt.tight_layout(); plt.show()

In [ ]:
# 3-4: lower/upper bounds (H, S, V)
RESTRICTIVE = (np.array([25, 200, 200]), np.array([28, 255, 255]))   # <-- TUNE: deliberately too tight
FINAL       = (np.array([18,  90,  90]), np.array([38, 255, 255]))   # <-- TUNE: captures the whole object
# NOTE red wraps around H=0/179 -> build two masks and cv2.bitwise_or them.

def color_mask(hsv, lo_hi):
    return cv2.inRange(hsv, lo_hi[0], lo_hi[1])

m_r, m_f = color_mask(hsv4, RESTRICTIVE), color_mask(hsv4, FINAL)
e_r = cv2.bitwise_and(bgr4, bgr4, mask=m_r)                # 6. extract colour from original
e_f = cv2.bitwise_and(bgr4, bgr4, mask=m_f)

print(f"Restrictive mask covers {fg_percent(m_r)}% of image | Final mask covers {fg_percent(m_f)}%")
show([bgr4, m_r, e_r, m_f, e_f],
     ["Original", f"Restrictive mask\n{RESTRICTIVE[0]}-{RESTRICTIVE[1]}", "Restrictive extraction",
      f"Final mask\n{FINAL[0]}-{FINAL[1]}", "Final extraction"],
     cols=5, size=3.6, suptitle="Task 4 - HSV colour segmentation", save="task4_comparison.png")
save_img("task4_final_mask.png", m_f); save_img("task4_final_extraction.png", e_f)

**Why the restrictive mask fails:** an object's colour is not one HSV value. Shading, highlights, reflections and shadows spread its pixels across a *range* of **S** and **V** (and slightly H). A narrow window keeps only the few pixels that match exactly → fragmented, incomplete mask (missing shadowed panels, highlights, edges). Widening the bounds (esp. S and V lower limits) recovers them; widen too far and background of similar hue leaks in. Hue carries the "colour identity" and is stable under lighting — that's why HSV beats thresholding intensity.

---
# Task 5 — Canny edge detection (hysteresis)
**Image:** a mechanical part / object with strong *and* faint edges.

In [ ]:
EDGE_URL = ""   # <-- object image link

edge_gray = resize_max(load_image(EDGE_URL, "gray"))
blur5 = cv2.GaussianBlur(edge_gray, (5, 5), 1.4)            # smooth first -> fewer false edges

PAIRS = [(50, 100), (50, 200), (50, 300)]                   # <-- TUNE (low fixed, high increasing)
edges = [cv2.Canny(blur5, lo, hi) for lo, hi in PAIRS]
show([edge_gray] + edges,
     ["Original"] + [f"Edge Result {i+1}  (low={lo}, high={hi})" for i, (lo, hi) in enumerate(PAIRS)],
     cols=4, size=4, suptitle="Task 5 - Canny with three threshold pairs", save="task5_comparison.png")
display(pd.DataFrame([{"pair": f"{lo}/{hi}", "edge pixels": int((e > 0).sum()), "edge %": fg_percent(e)}
                      for (lo, hi), e in zip(PAIRS, edges)]))
save_img("task5_final_edges.png", edges[1])

In [ ]:
# Visualise the hysteresis idea: strong / weak / rejected gradient pixels for a chosen pair (pre-NMS approximation)
LO, HI = PAIRS[1]
gx = cv2.Sobel(blur5, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(blur5, cv2.CV_32F, 0, 1, ksize=3)
mag = np.abs(gx) + np.abs(gy)                                # Canny's default (L1) magnitude
vis = cv2.cvtColor(edge_gray, cv2.COLOR_GRAY2BGR)
vis[(mag >= LO) & (mag < HI)] = (0, 255, 255)                # yellow = weak (kept only if connected to strong)
vis[mag >= HI] = (0, 200, 0)                                 # green  = strong (always kept)

# edges that exist at the lowest high-threshold but vanish at the highest = weak / possibly unwanted
lost = cv2.subtract(edges[0], edges[-1])
show([vis, edges[1], lost],
     [f"Strong (green) / Weak (yellow), low={LO}, high={HI}", "Canny result", f"Present at high={PAIRS[0][1]} but LOST at high={PAIRS[-1][1]}"],
     cols=3, size=4.6, save="task5_hysteresis.png")

**Identify (inspect the figures above and keep what matches):**
- **Strong edges:** outer silhouette / high-contrast boundaries — survive every pair.
- **Weak edges:** low-contrast boundaries (soft shadows, faint inner contours) — appear only when `high` is low; they survive only if connected to a strong edge.
- **Missing edges:** real boundaries that disappear at the highest `high` (see the "lost" panel).
- **Unwanted edges:** texture, noise, specular glints, background clutter — dominate at low thresholds.

**How the thresholds work (hysteresis):** gradient > `high` ⇒ definitely edge; gradient < `low` ⇒ discarded; between ⇒ kept **only if connected to a strong edge**. With `low` fixed, raising `high` shrinks the set of seeds, so fewer weak chains get "rescued": cleaner but with gaps. Lowering `low` (with `high` fixed) lets chains extend further and joins broken edges, but adds clutter.

---
# Task 6 — Region growing (brain MRI)
**Image:** brain MRI (grayscale). Seeds are `(x, y)` = (column, row).

In [ ]:
MRI_URL = ""   # <-- MRI image link

mri = resize_max(load_image(MRI_URL, "gray"), 512)
mri_s = cv2.medianBlur(mri, 5)                                # light denoise so single noisy pixels don't stop growth
h6, w6 = mri.shape

def region_grow(gray, seed, thr, connectivity=8, mode="seed"):
    """BFS region growing. mode='seed': compare with seed intensity; 'mean': compare with running region mean."""
    h, w = gray.shape; x0, y0 = seed
    assert 0 <= x0 < w and 0 <= y0 < h, "seed outside image"
    g = gray.astype(np.int16)
    mask = np.zeros((h, w), np.uint8); seen = np.zeros((h, w), bool)
    ref, total, count = int(g[y0, x0]), int(g[y0, x0]), 1
    nb = [(-1, 0), (1, 0), (0, -1), (0, 1)] + ([(-1, -1), (-1, 1), (1, -1), (1, 1)] if connectivity == 8 else [])
    q = deque([(x0, y0)]); seen[y0, x0] = True; mask[y0, x0] = 255
    while q:
        cx, cy = q.popleft()
        for dx, dy in nb:
            nx, ny = cx + dx, cy + dy
            if 0 <= nx < w and 0 <= ny < h and not seen[ny, nx]:
                seen[ny, nx] = True
                r = ref if mode == "seed" else total / count
                if abs(int(g[ny, nx]) - r) <= thr:
                    mask[ny, nx] = 255; q.append((nx, ny)); total += int(g[ny, nx]); count += 1
    return mask

def overlay(gray, mask, color=(0, 0, 255), a=0.5):
    b = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR); o = b.copy(); o[mask > 0] = color
    return cv2.addWeighted(o, a, b, 1 - a, 0)

# look at the image with coordinates to choose seeds
plt.figure(figsize=(6, 6)); plt.imshow(mri, cmap="gray"); plt.grid(alpha=.3)
plt.title("Pick seed (x, y) from the axes"); plt.show()

In [ ]:
SEEDS = [(int(w6*0.50), int(h6*0.50)), (int(w6*0.30), int(h6*0.40))]   # <-- TUNE: >= 2 seeds
THRS  = [5, 12, 25]                                                     # <-- TUNE: >= 3 thresholds

imgs, titles, rec, masks = [], [], [], {}
for si, seed in enumerate(SEEDS, 1):
    for t in THRS:
        m = region_grow(mri_s, seed, t)
        masks[(si, t)] = m
        o = overlay(mri, m); cv2.circle(o, seed, 5, (0, 255, 0), -1)
        imgs.append(o); titles.append(f"seed{si}={seed}  thr={t}")
        rec.append({"seed": f"S{si} {seed}", "seed intensity": int(mri_s[seed[1], seed[0]]),
                    "threshold": t, "region area %": fg_percent(m)})
show(imgs, titles, cols=len(THRS), size=3.8, suptitle="Task 6 - region growing (rows: seeds, cols: thresholds)", save="task6_grid.png")
display(pd.DataFrame(rec))
save_img("task6_final_mask.png", masks[(1, THRS[1])])           # <-- choose your final (seed, threshold)

**Why can changing the seed point significantly change the segmented region?**
Growth only follows pixels whose intensity is within the threshold of the *seed's* intensity **and** that are connected to it. A different seed (a) has a different reference intensity, so a different set of pixels qualifies, and (b) may sit in a different connected structure, or in a noisy/boundary pixel whose value isn't representative of the tissue. Regions with similar intensity elsewhere in the brain are only included if they're reachable through qualifying neighbours — which a global threshold would include regardless. Small thresholds → under-segmentation; large thresholds → leakage across boundaries into neighbouring tissue.

---
# Task 7 — Marker-based watershed (touching coins)
**Image:** the OpenCV watershed coins image (touching coins). The pipeline is a function so Task 8 can re-run it with different distance thresholds.

In [ ]:
COINS7_URL = ""   # <-- touching-coins image link

coins7 = resize_max(load_image(COINS7_URL, "color"), 800)

def run_watershed(bgr, dist_frac=0.7, blur=5, open_iter=2, dilate_iter=3):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)                                    # 1. preprocessing
    gray = cv2.GaussianBlur(gray, (blur, blur), 0)
    _, thr = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)       # 2. thresholding
    thr = fix_polarity(thr)                                                         #    objects white
    k = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(thr, cv2.MORPH_OPEN, k, iterations=open_iter)        # 3. noise removal
    sure_bg = cv2.dilate(opening, k, iterations=dilate_iter)                        # 4. sure background
    dist = cv2.distanceTransform(opening, cv2.DIST_L2, 5)                           # 5. distance transform
    dist_t = dist_frac * dist.max()
    sure_fg = np.uint8(dist > dist_t) * 255                                         # 6. sure foreground
    unknown = cv2.subtract(sure_bg, sure_fg)                                        # 7. unknown region
    n_fg, markers = cv2.connectedComponents(sure_fg)                                # 8. marker labelling
    markers = markers + 1                      # background = 1 (not 0, 0 is reserved for "unknown")
    markers[unknown == 255] = 0
    markers = cv2.watershed(bgr.copy(), markers)                                    # 9. watershed
    final = bgr.copy(); final[markers == -1] = (0, 0, 255)                          # 10. boundaries in red
    regions = len(set(np.unique(markers)) - {-1, 1})
    return dict(thr=thr, sure_bg=sure_bg, dist=dist, dist_t=dist_t, sure_fg=sure_fg, unknown=unknown,
                markers=markers, final=final, n_fg=n_fg - 1, n_regions=regions, n_connected=cv2.connectedComponents(opening)[0] - 1)

DIST_FRAC = 0.7     # <-- TUNE
r7 = run_watershed(coins7, DIST_FRAC)
dist_vis = cv2.normalize(r7["dist"], None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

show([coins7, r7["thr"], r7["sure_bg"], dist_vis, r7["sure_fg"], r7["unknown"], r7["final"]],
     ["Original", "Threshold (Otsu)", "Sure Background", "Distance Transform", "Sure Foreground", "Unknown Region", "Final Watershed"],
     cols=4, size=3.6, suptitle="Task 7 - watershed pipeline", save="task7_pipeline.png")
print(f"Connected white blobs after threshold: {r7['n_connected']}  ->  after watershed: {r7['n_regions']} separate coins")
save_img("task7_final_watershed.png", r7["final"])

**Visual inspection:** check the red boundary lines — each touching pair/cluster should now be split by a line along the neck between coins, and the count after watershed should be larger than the raw connected-component count (printed above). If coins are still merged → distance threshold too low (Task 8); if single coins are cut in two → too high or noisy markers.

---
# Task 8 — Tuning the distance-transform threshold
Reuses `run_watershed` and the Task 7 coins (set `COINS8_URL` only if different).

In [ ]:
COINS8_URL = ""     # <-- "" reuses Task 7's image
coins8 = coins7 if COINS8_URL == "" else resize_max(load_image(COINS8_URL, "color"), 800)

EXPERIMENTS = [0.2, 0.45, 0.7, 0.9]    # <-- TUNE: fractions of max distance (>= 4)
EXPECTED_COINS = None                   # <-- optional: true number of coins you count by eye -> auto "Observed Result"
OBSERVED = {}                           # <-- optional manual override, e.g. {1: "merged coins", 3: "good"}

rows, imgs, ttl = [], [], []
for i, f in enumerate(EXPERIMENTS, 1):
    r = run_watershed(coins8, f)
    if i in OBSERVED: obs = OBSERVED[i]
    elif EXPECTED_COINS is None: obs = "(fill in by eye)"
    elif r["n_regions"] < EXPECTED_COINS: obs = "under-separated (coins merged)"
    elif r["n_regions"] > EXPECTED_COINS: obs = "over-split / spurious markers"
    else: obs = "matches expected count"
    rows.append({"Experiment": i, "Distance Threshold": f"{f:.2f} x max = {r['dist_t']:.2f}",
                 "Foreground markers": r["n_fg"], "Regions": r["n_regions"], "Observed Result": obs})
    imgs += [r["sure_fg"], r["final"]]; ttl += [f"Exp {i}: sure FG (frac={f})", f"Exp {i}: watershed ({r['n_regions']} regions)"]

show(imgs, ttl, cols=4, size=3.4, suptitle="Task 8 - sure foreground / result per experiment", save="task8_experiments.png")
table8 = pd.DataFrame(rows); display(table8)
table8.to_csv(os.path.join(OUT_DIR, "task8_table.csv"), index=False)

**Most meaningful parameter range:** pick the experiments where *markers ≈ regions ≈ true coin count* and the red boundaries sit on the necks between coins. Typically that is a **mid range (≈ 0.4 – 0.7 × max distance)**.
- **Too low:** sure-foreground blobs of touching coins fuse into one marker → merged coins (fewer markers than coins).
- **Too high:** only the very centres survive; small/off-centre coins lose their marker entirely (missed) or a coin gets several tiny markers (over-split).
Confirm the range with your own table.

---
# Task 9 — K-Means colour segmentation
Workflow from the manual: reshape → float32 → `cv2.kmeans` → centres → reconstruct.

In [ ]:
WILD_URL = ""   # <-- complex colour image link

wild = resize_max(load_image(WILD_URL, "color"), 600)     # smaller = faster k-means

def kmeans_seg(bgr, K, attempts=10, seed=42):
    cv2.setRNGSeed(seed)
    Z = bgr.reshape((-1, 3)).astype(np.float32)                              # reshape + float32
    crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    _, labels, centers = cv2.kmeans(Z, K, None, crit, attempts, cv2.KMEANS_PP_CENTERS)
    centers = np.uint8(centers)
    seg = centers[labels.flatten()].reshape(bgr.shape)                       # reconstruct from centres
    return seg, labels.flatten(), centers

Ks = (2, 4, 6)
segs, rows = {}, []
n_orig = len(np.unique(wild.reshape(-1, 3), axis=0))
for K in Ks:
    seg, lab, cen = kmeans_seg(wild, K)
    segs[K] = (seg, lab, cen)
    mse = float(np.mean((wild.astype(np.float32) - seg.astype(np.float32))**2))
    share = np.bincount(lab, minlength=K) / lab.size * 100
    rows.append({"K": K, "clusters": K, "unique colours": f"{n_orig} -> {len(np.unique(seg.reshape(-1,3), axis=0))}",
                 "MSE vs original": round(mse, 1),
                 "cluster share % (BGR centre)": "; ".join(f"{s:.0f}% {tuple(int(v) for v in c)}" for s, c in zip(share, cen))})

show([wild] + [segs[K][0] for K in Ks], ["Original"] + [f"K={K}" for K in Ks],
     cols=4, size=4, suptitle="Task 9 - K-Means segmentation", save="task9_comparison.png")
display(pd.DataFrame(rows))
for K in Ks: save_img(f"task9_k{K}.png", segs[K][0])

# palettes: which colour region each cluster represents
fig, axs = plt.subplots(1, len(Ks), figsize=(12, 1.6))
for ax, K in zip(axs, Ks):
    cen, lab = segs[K][2], segs[K][1]
    order = np.argsort(-np.bincount(lab, minlength=K))
    bar = np.hstack([np.full((40, int(300*np.mean(lab == i)) + 1, 3), cen[i][::-1], np.uint8) for i in order])
    ax.imshow(bar); ax.set_title(f"K={K} palette (width = area share)"); ax.axis("off")
plt.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "task9_palettes.png"), bbox_inches="tight", dpi=150); plt.show()

**Record per K (fill from the table/figure):**
| K | Visual difference from original | Colour simplification | Major regions represented |
|---|---|---|---|
| 2 | Almost a binary-looking split; detail and gradients gone | Highest (2 colours) | Typically light vs dark (e.g. sky/background vs animal/ground) |
| 4 | Main structure kept, posterised look | Strong | Sky, vegetation, animal body, shadows… (name yours) |
| 6 | Closest to original; subtle tones separated | Lowest of the three, still posterised | Adds highlight/shadow or fur-tone clusters (name yours) |

As K grows, MSE drops and the image looks more natural, but regions become less "simple". K-Means uses **colour only** (no spatial info), so separate objects with similar colours end up in the same cluster.

---
# Task 10 — Segmentation engineer: pick 3 methods for an unknown image
Choose a challenging image (or reuse one above). Pick **three methods** in `CHOSEN`; all eight are implemented with editable parameters.

In [ ]:
UNK_URL = ""   # <-- challenging image link ("" = upload)
img10 = resize_max(load_image(UNK_URL, "color"), 600)
gray10 = cv2.cvtColor(img10, cv2.COLOR_BGR2GRAY)
h10, w10 = gray10.shape

PARAMS = {
    "global":    dict(t=127),
    "adaptive":  dict(block=31, c=10),
    "otsu":      dict(blur=5),
    "color":     dict(lo=(18, 90, 90), hi=(38, 255, 255)),
    "canny":     dict(lo=50, hi=150),
    "region":    dict(seed=(w10//2, h10//2), thr=15),
    "watershed": dict(dist_frac=0.6),
    "kmeans":    dict(K=4),
}

def m_global(p):    return cv2.threshold(gray10, p["t"], 255, cv2.THRESH_BINARY)[1]
def m_adaptive(p):  return cv2.adaptiveThreshold(gray10, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, p["block"], p["c"])
def m_otsu(p):      return cv2.threshold(cv2.GaussianBlur(gray10, (p["blur"],)*2, 0), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
def m_color(p):     return cv2.inRange(cv2.cvtColor(img10, cv2.COLOR_BGR2HSV), np.array(p["lo"]), np.array(p["hi"]))
def m_canny(p):     return cv2.Canny(cv2.GaussianBlur(gray10, (5, 5), 1.4), p["lo"], p["hi"])
def m_region(p):    return region_grow(cv2.medianBlur(gray10, 5), p["seed"], p["thr"])
def m_watershed(p): return run_watershed(img10, p["dist_frac"])["final"]
def m_kmeans(p):    return kmeans_seg(img10, p["K"])[0]
METHODS10 = dict(zip(PARAMS, [m_global, m_adaptive, m_otsu, m_color, m_canny, m_region, m_watershed, m_kmeans]))

CHOSEN = ["otsu", "adaptive", "kmeans"]       # <-- TUNE: pick 3 of the 8 keys above
outs = [METHODS10[k](PARAMS[k]) for k in CHOSEN]
show([img10] + outs, ["Original"] + [f"Method {i+1}: {k}\n{PARAMS[k]}" for i, k in enumerate(CHOSEN)],
     cols=4, size=4, suptitle="Task 10 - Original | Method 1 | Method 2 | Method 3", save="task10_comparison.png")
for k, o in zip(CHOSEN, outs): save_img(f"task10_{k}.png", o)

In [ ]:
# Which parameter has the greatest effect? Sweep each chosen method's key parameter and measure how much the output changes.
SWEEPS = {   # method -> (param, values)             <-- TUNE
    "global":    ("t", [60, 127, 200]),
    "adaptive":  ("c", [2, 10, 25]),
    "otsu":      ("blur", [1, 5, 15]),
    "color":     ("lo", [(10, 60, 60), (18, 90, 90), (25, 150, 150)]),
    "canny":     ("hi", [100, 200, 300]),
    "region":    ("thr", [5, 15, 40]),
    "watershed": ("dist_frac", [0.2, 0.6, 0.9]),
    "kmeans":    ("K", [2, 4, 8]),
}
rows = []
for k in CHOSEN:
    name, vals = SWEEPS[k]
    base = METHODS10[k](PARAMS[k]).astype(np.float32)
    diffs = []
    for v in vals:
        p = dict(PARAMS[k]); p[name] = v
        diffs.append(float(np.mean(np.abs(METHODS10[k](p).astype(np.float32) - base))))
    rows.append({"method": k, "parameter swept": name, "values": vals, "mean |change| vs default": round(max(diffs), 1)})
display(pd.DataFrame(rows))

### Per-method analysis (fill in from YOUR observed results — the grader wants observations, not textbook answers)
| | Method 1 | Method 2 | Method 3 |
|---|---|---|---|
| **1. Assumption about the image** | | | |
| **2. Region/object correctly identified** | | | |
| **3. Part incorrectly segmented** | | | |
| **4. Parameter with greatest effect** *(use the sweep table)* | | | |

*Reference assumptions:* **Otsu** – two-class bimodal histogram, even lighting · **Adaptive** – illumination varies slowly, objects differ locally from surroundings · **Global** – uniform lighting + good contrast · **Colour/HSV** – object has distinct hue · **Canny** – object boundaries = intensity jumps · **Region growing** – target is a connected area of similar intensity & seed is representative · **Watershed** – touching objects are roughly convex blobs with distance-transform peaks · **K-Means** – regions are separable by colour alone.

### Technical conclusion (write after viewing the comparison)
> *"For this image, **<method>** produced the most meaningful regions because <image property, e.g. strong bimodal contrast / uneven lighting / distinct hue>. **<method X>** failed on <region> because <its assumption was violated>. The most influential parameter was <param>."*

---
## Export all outputs

In [ ]:
zpath = "/content/lab05_outputs.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUT_DIR)): z.write(os.path.join(OUT_DIR, f), f)
print(len(os.listdir(OUT_DIR)), "files zipped ->", zpath)
try:
    from google.colab import files; files.download(zpath)
except Exception: pass